In [ ]:
import os

import numpy as np

from scripts.data_loader import load_matlab_file_as_df
from scripts.utils import RF_PARAM, extract_unique_npcis

# source file
BASE_DIR = "data/"
FULL_DATA_SET = "Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat"
filename = os.path.join(BASE_DIR, FULL_DATA_SET)

# params
operator_choice = np.array([1, 10, 88])
rf_param = RF_PARAM.NSINR
# load the dataset as pandas dataframe

df = load_matlab_file_as_df(
    filename=filename,
    dataset='dataSet_smooth',  # dataSet, dataSet_interp or dataSet_smooth
    usecols=['lat', 'lng', 'measurements_matrix', 'campaign_id']
)
# Using 'Total Smoothing', both interpolation and smoothing applied for both RPs ant TPs
df_smooth = df.copy()

# Use a series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

test_campaign = 1

## Data preprocessing
1. Clustering the RP's with k-means based on lat,lng coordinates
2. Training a Random Forest classifier using NSINR as the feature to predict the cluster

In [ ]:
from scripts.plotting import make_boxplot
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from scripts.weighted_coverage import create_point_matrix
from sklearn.ensemble import RandomForestClassifier
from scripts.clustering import train_kmeans

rf_param = RF_PARAM.NSINR
operator_choice = np.array([1, 10, 88])
unique_npcis = extract_unique_npcis(df, operator_choice)

# experiment setup
n_runs = 40
max_clusters = 20
cluster_range = range(1, max_clusters + 1)

accuracy_dict = {n_clusters: [] for n_clusters in cluster_range}
recall_dict = {n_clusters: [] for n_clusters in cluster_range}

for run in range(n_runs):
    print(f'\r🔄 {run + 1}/{n_runs} runs', end='')
    for n_clusters in cluster_range:
        df_tmp = df.copy()
        kMeans, cluster_labels = train_kmeans(df_tmp, n_clusters, random_seeds[n_clusters])
        df_tmp['cluster'] = cluster_labels

        test_mask = np.random.rand(len(df_tmp)) <= 0.3
        tp = df_tmp[test_mask].copy()
        rp = df_tmp[~test_mask].copy()

        m_tp, _ = create_point_matrix(tp, unique_npcis, rf_param)
        m_rp, _ = create_point_matrix(rp, unique_npcis, rf_param)

        rf_model = RandomForestClassifier(n_estimators=100, random_state=random_seeds[n_clusters])
        rf_model.fit(m_rp, rp['cluster'])

        tp['predicted'] = rf_model.predict(m_tp)

        # get the accuracy
        accuracy = accuracy_score(tp['cluster'], tp['predicted'])
        accuracy_dict[n_clusters].append(accuracy * 100)

        # get the average recall
        report = classification_report(tp['cluster'], tp['predicted'], output_dict=True)
        recall_values = {label: metrics['recall'] for label, metrics in report.items() if label != 'accuracy'}
        recall_dict[n_clusters].append(sum(recall_values.values()) / len(recall_values) * 100)

print(f'\r✅ ')

accuracy_df = pd.DataFrame(accuracy_dict)
make_boxplot(accuracy_df, 'Random Forest Accuracy', 'Clusters in KMeans', 'Accuracy (%)')

recall_df = pd.DataFrame(recall_dict)
make_boxplot(recall_df, 'Random Forest Mean Recall', 'Clusters in KMeans', 'Recall (%)')

In [ ]:
make_boxplot(accuracy_df, 'Random Forest Classifier Accuracy with NSINR (40 runs)', 'Clusters in KMeans',
             'Accuracy (%)')
make_boxplot(recall_df, 'Random Forest Classifier Mean Recall with NSINR (40 runs)', 'Clusters in KMeans', 'Recall (%)')